# MLP Training — Credit Risk Default Prediction

Binary classification with PyTorch. Class imbalance handled via `pos_weight` in BCE loss.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# =============================================================================
# Environment setup — works both locally and in Google Colab
# =============================================================================
try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected.")
    print("Upload train.csv, val.csv, test.csv to /content/ or set DATA_PATH.")
    DATA_PATH = "/content"
except ImportError:
    IN_COLAB = False
    DATA_PATH = "../../data/processed"
    print("Local environment detected.")

OUT_DIR = "./output"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUT_DIR:   {OUT_DIR}")

In [ ]:
# --- Load processed data ---
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
val_df   = pd.read_csv(f"{DATA_PATH}/val.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")

print(f"Train: {train_df.shape}  |  Val: {val_df.shape}  |  Test: {test_df.shape}")
print(f"Train default rate: {train_df['DEFAULT_OCT'].mean():.4f}")
print(f"Val   default rate: {val_df['DEFAULT_OCT'].mean():.4f}")
print(f"Test  default rate: {test_df['DEFAULT_OCT'].mean():.4f}")

In [ ]:
# --- PyTorch Dataset ---
class CreditRiskDataset(Dataset):
    def __init__(self, df):
        self.X = torch.tensor(df.drop(columns=['DEFAULT_OCT']).values, dtype=torch.float32)
        self.y = torch.tensor(df['DEFAULT_OCT'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = CreditRiskDataset(train_df)
val_ds   = CreditRiskDataset(val_df)
test_ds  = CreditRiskDataset(test_df)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Input features: {train_ds.X.shape[1]}")
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

In [ ]:
# --- MLP Model ---
class CreditRiskMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32, 16], dropout_rates=[0.3, 0.3, 0.2]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hd, dr in zip(hidden_dims, dropout_rates):
            layers.append(nn.Linear(prev_dim, hd))
            layers.append(nn.BatchNorm1d(hd))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dr))
            prev_dim = hd
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev_dim, 1)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

input_dim = train_ds.X.shape[1]
print(f"Input dimension: {input_dim}")

In [ ]:
# =============================================================================
# Focal Loss — penalises minority-class mispredictions
# =============================================================================
#   FL(p_t) = -α_t · (1 - p_t)^γ · log(p_t)
#
#  α (alpha): class weight — up-weights the minority class
#  γ (gamma): focusing parameter — keeps loss high for confidently-wrong predictions
#
# A true defaulter predicted at 5% probability gets MUCH higher loss than
# one at 70%. The model can't coast on easy majority-class examples.

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.78, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)       # p of the TRUE class
        focal_weight = (1 - p_t) ** self.gamma              # down-weight easy examples
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * focal_weight * bce).mean()

In [ ]:
# =============================================================================
# Build final model — use best params from tuning if available
# =============================================================================

if BEST_PARAMS:
    arch_map = {
        'small': [64, 32, 16], 'medium': [128, 64, 32],
        'wide': [256, 128],    'deep': [128, 64, 32, 16],
    }
    best_hidden = arch_map[BEST_PARAMS['architecture']]
    n_layers = len(best_hidden)
    best_dropout = [BEST_PARAMS[f'dropout_{i}'] for i in range(n_layers)]

    model = CreditRiskMLP(input_dim, best_hidden, best_dropout).to(device)
    criterion = FocalLoss(
        alpha=BEST_PARAMS['focal_alpha'],
        gamma=BEST_PARAMS['focal_gamma']
    )
    optimizer = optim.Adam(
        model.parameters(),
        lr=BEST_PARAMS['lr'],
        weight_decay=BEST_PARAMS['weight_decay']
    )
    BATCH_SIZE = BEST_PARAMS['batch_size']

    print(f"Rebuilt with tuned params: arch={best_hidden}, lr={BEST_PARAMS['lr']:.2e}, "
          f"batch={BATCH_SIZE}, gamma={BEST_PARAMS['focal_gamma']:.2f}")
else:
    # Default fallback (when RUN_TUNING = False)
    pos_count = train_df['DEFAULT_OCT'].sum()
    neg_count = len(train_df) - int(pos_count)
    alpha = neg_count / len(train_df)

    model = CreditRiskMLP(input_dim).to(device)
    criterion = FocalLoss(alpha=alpha, gamma=2.0)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    BATCH_SIZE = 256

    print(f"Using defaults: arch=[64,32,16], lr=1e-3, batch=256, gamma=2.0")

# Rebuild dataloaders with the (possibly updated) BATCH_SIZE
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} trainable")

# Hyperparameter Tuning with Optuna

Run the cell below to search for optimal hyperparameters. Skips tuning in quick-test mode.

In [ ]:
# --- Training loop ---
EPOCHS = 100
PATIENCE = 20

history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * Xb.size(0)
    train_loss /= len(train_ds)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * Xb.size(0)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    val_loss /= len(val_ds)
    val_auc = roc_auc_score(all_labels, all_probs)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    scheduler.step(val_loss)

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_auc: {val_auc:.4f}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (best: {best_epoch}, val_loss={best_val_loss:.4f})")
        break

if patience_counter < PATIENCE:
    print(f"\nTraining complete. Best epoch: {best_epoch}, val_loss: {best_val_loss:.4f}")

In [ ]:
# --- Training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.axvline(x=best_epoch - 1, color='gray', linestyle='--', alpha=0.5,
           label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()

ax2.plot(history['val_auc'], color='green')
ax2.axvline(x=best_epoch - 1, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Validation AUC-ROC')

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate on test set ---
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", weights_only=True))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = model(Xb)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds = (all_probs >= 0.5).astype(int)

auc = roc_auc_score(all_labels, all_probs)
print(f"Test AUC-ROC: {auc:.4f}")
print(f"\n{classification_report(all_labels, all_preds, target_names=['Non-Default', 'Default'])}")

In [ ]:
# --- Confusion Matrix & ROC Curve ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix (Test, threshold=0.5)')

RocCurveDisplay.from_predictions(all_labels, all_probs, ax=ax2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax2.set_title(f'ROC Curve (AUC = {auc:.4f})')

plt.tight_layout()
plt.show()

In [ ]:
# --- Save results & download (Colab) ---
pd.DataFrame({
    'auc_roc': [auc],
    'best_val_loss': [best_val_loss],
    'best_epoch': [best_epoch],
}).to_csv(f"{OUT_DIR}/results.csv", index=False)

print(f"Artifacts saved to {OUT_DIR}/")
print(f"  best_model.pt, results.csv")

if IN_COLAB:
    from google.colab import files
    import shutil
    shutil.make_archive(f"{OUT_DIR}/model", 'zip', OUT_DIR)
    print(f"\nDownloading model.zip...")
    files.download(f"{OUT_DIR}/model.zip")

In [ ]:
# --- Probability distribution by class ---
fig, ax = plt.subplots(figsize=(10, 5))

for label, color, name in [(0, 'blue', 'Non-Default'), (1, 'red', 'Default')]:
    mask = all_labels == label
    sns.kdeplot(all_probs[mask], fill=True, alpha=0.3, color=color, label=name, ax=ax)

ax.axvline(x=0.5, color='gray', linestyle='--', label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability of Default')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distribution by Actual Class')
ax.legend()
plt.show()